# Native MIS 小规模求解验证

这本 notebook 验证 `mis.v1 -> MIS solver -> mis-result.v1` 的原生图路径。Exact、Greedy、独立 oracle 与 ProblemCase exact promotion 都调用脚本中的实现；notebook 只负责组装小案例、展示和断言。

## 1. 定位仓库并导入

In [ ]:
import copy
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_mis, validate_mis_result
from lib.solvers.mis import ExactMisSolver, GreedyMisSolver
from problem import (
    ProblemArtifact,
    ProblemCase,
    TaskDefinition,
    solve_native_problem_task,
    validate_problem_case,
)
from tests.oracles.mis import enumerate_mis
from tests.oracles.numbers import public_json_number

## 2. 读取 canonical MIS

直接读取契约示例。顶点索引是稳定身份，边采用 `u < v` 且字典序排列的规范形式。

In [ ]:
example_path = PROJECT_ROOT / "contracts" / "examples" / "mis.v1.example.json"
problem = json.loads(example_path.read_text(encoding="utf-8"))
original_problem = copy.deepcopy(problem)

validate_mis(problem)
print("Problem:", problem["problem_id"])
print("Objective:", problem["objective"]["kind"])
print("Vertices:", [vertex["name"] for vertex in problem["vertices"]])
print("Edges:", problem["edges"])

## 3. 独立 oracle、Exact 与 Greedy

`tests.oracles.mis` 不导入 production solver 或 evaluator。Exact 必须与独立枚举的目标值和规范见证一致；Greedy 只承诺返回经过契约重算的可行 incumbent。

In [ ]:
oracle_row = enumerate_mis(problem)[0]
oracle_objective = public_json_number(oracle_row["objective_exact"])

exact_result = ExactMisSolver().solve(problem)
greedy_result = GreedyMisSolver().solve(problem)
validate_mis_result(problem, exact_result)
validate_mis_result(problem, greedy_result)

assert exact_result["status"] == "optimal"
assert exact_result["selected_vertices"] == oracle_row["selected_vertices"]
assert exact_result["objective_value"] == oracle_objective
assert exact_result["bounds"]["incumbent_lower_bound"] == exact_result["bounds"]["optimum_upper_bound"]
assert greedy_result["status"] == "feasible"
assert greedy_result["feasible"] is True
assert problem == original_problem

print("Oracle:", oracle_row["selected_vertices"], oracle_objective)
print("Exact:", exact_result["selected_vertices"], exact_result["proof"])
print("Greedy:", greedy_result["selected_vertices"], greedy_result["metrics"])

## 4. Maximum-weight MIS

同一张路径图改为显式权重后，目标由集合大小切换为所选顶点权重之和。

In [ ]:
weighted_problem = copy.deepcopy(problem)
weighted_problem["problem_id"] = "weighted-path-four"
weighted_problem["objective"] = {"kind": "maximum-weight"}
weights = [3, 8, 4, 3]
weighted_problem["vertices"] = [
    {"index": vertex["index"], "name": vertex["name"], "weight": weights[vertex["index"]]}
    for vertex in weighted_problem["vertices"]
]

weighted_oracle = enumerate_mis(weighted_problem)[0]
weighted_exact = ExactMisSolver().solve(weighted_problem)
weighted_greedy = GreedyMisSolver().solve(weighted_problem)

assert weighted_exact["selected_vertices"] == weighted_oracle["selected_vertices"] == [1, 3]
assert weighted_exact["objective_value"] == 11
assert weighted_greedy["feasible"] is True

print("Weighted exact:", weighted_exact["selected_vertices"], weighted_exact["objective_value"])
print("Weighted greedy:", weighted_greedy["selected_vertices"], weighted_greedy["objective_value"])

## 5. Fixed-value 不可行证书

相邻顶点同时固定为 1 时，不存在满足 hard assignments 的独立集；该边本身就是直接不可行证据。

In [ ]:
infeasible_problem = copy.deepcopy(problem)
infeasible_problem["problem_id"] = "fixed-edge-conflict"
infeasible_problem["fixed_values"] = [
    {"index": 0, "value": 1},
    {"index": 1, "value": 1},
]

infeasible_result = ExactMisSolver().solve(infeasible_problem)
validate_mis_result(infeasible_problem, infeasible_result)

assert infeasible_result["status"] == "infeasible"
assert infeasible_result["selected_vertices"] is None
assert infeasible_result["proof"]["kind"] == "fixed-edge-conflict"

print("Infeasibility proof:", infeasible_result["proof"])

## 6. ProblemCase 原生执行与 exact promotion

`mis-result.v1` 的 `optimal` 是 solver 声明。写入 task 的 `exact=True` 前，ProblemCase 执行桥会在规模上限内独立枚举 canonical MIS。

In [ ]:
case = ProblemCase(
    problem_id=problem["problem_id"],
    artifacts=(
        ProblemArtifact(
            artifact_id="mis",
            representation="mis.v1",
            payload=problem,
        ),
    ),
    tasks=(
        TaskDefinition(
            task_id="maximum-independent-set",
            canonical_artifact_id="mis",
            sense="maximize",
            solution_representation="vertex-index-set.v1",
            task_type="maximum-independent-set",
        ),
    ),
)

record = solve_native_problem_task(
    case,
    task_id="maximum-independent-set",
    artifact_id="mis",
    solver=ExactMisSolver(),
    update_best=True,
    exact_verification_max_variables=24,
)

assert record.canonical_solution == oracle_row["selected_vertices"]
assert record.canonical_objective_value == oracle_objective
assert record.exact_for_task is True
assert record.update.current.exact is True
assert record.update.current.metadata["exactness"]["independent_mis_optimality_verified"] is True
assert validate_problem_case(record.case, strict=True).fully_checked

print("Persistent exact:", record.exact_for_task)
print("Exactness evidence:", record.update.current.metadata["exactness"])

## 结论与边界

- `mis.v1` 与 `mis-result.v1` 保留图原生语义，见证是规范的顶点索引集，不是伪装成图问题的 QUBO bit vector。
- Exact 使用 branch-and-reduce，并以独立 oracle 交叉验证；`max_vertices` 是显式指数搜索安全上限。
- Greedy 加局部交换只返回 `feasible`，即使命中同一个最优见证也不声称最优性。
- ProblemCase exact promotion 依靠第二条独立枚举路径；超过复核上限时可保留 incumbent，但保持非 exact。
- timeout、随机图交叉验证、配置错误和输入不可变性由自动化测试覆盖，notebook 不复制这些测试 helper。